# Add the new history models to Run 1

**Locally: select your gpu_env kernel, restart it, then run all cells in order.** The paths below are preset to the extracted `Run_1_results` folder beside this notebook.

This notebook loads the existing 50,000-person cohort and its original train/validation/test split, copies the eight trained models, and reuses all 460 saved prediction files. It trains only **DynamicDeepHit**, **SurvTransformer** (SurvTRACE history adaptation with continuous PCH likelihood), and **SurvTransformer-Discrete**. Old models are relabeled **DynamicDeepHIt-Lite** and **SurvTransformer-Lite**. The Glazier models and their seven sampling methods are retained.

M=10,000, sampling seeds, evaluation patients and landmarks come directly from the previous run. No old rollout generation is needed. New models use every state observed through the landmark; neither future states nor the reference curves enter training/model selection. Output goes to a separate folder. Full new-model training starts only in the cell labeled **Train only the three new models**.

For Colab, upload this notebook with `Glazier_Weibull_HardRule24_Reuse_v3.zip`; make the old results available separately, preferably on Drive. Set the two paths below to the extracted old results and a persistent new output folder. The source ZIP does not include your old result files.

In [ ]:
import os, sys, datetime
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
if not hasattr(datetime,"UTC"): datetime.UTC=datetime.timezone.utc
IN_COLAB="google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable,"-m","pip","-q","install","lifelines==0.30.3","matplotlib","pandas"])
from pathlib import Path
import zipfile
BUNDLE=Path.cwd()  # adjust for a custom extraction directory
candidates=[BUNDLE,BUNDLE/"Glazier_Weibull_HardRule24",BUNDLE/"outputs/Glazier_Weibull_HardRule24"]
found=next((p for p in candidates if (p/"hard_rule_workflow.py").exists()),None)
if found is None and IN_COLAB:
    from google.colab import files
    print("Upload Glazier_Weibull_HardRule24_Reuse_v3.zip")
    uploaded=files.upload();archive=next((Path(k) for k in uploaded if k.endswith('.zip')),None)
    if archive is None: raise FileNotFoundError("Bundle ZIP required")
    dest=Path('/content/glazier_hard24').resolve();dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if any(not (dest/e.filename).resolve().is_relative_to(dest) for e in z.infolist()):raise ValueError('Unsafe ZIP path')
        z.extractall(dest)
    found=dest/'Glazier_Weibull_HardRule24'
if found is None:raise FileNotFoundError('Extract the bundle and set BUNDLE to its folder')
BUNDLE=found.resolve()
for parent in [BUNDLE,*BUNDLE.parents]:
    support=parent/'work/gpu_support'
    if support.exists():sys.path.insert(0,str(support));break
sys.path[:0]=[str(BUNDLE),str(BUNDLE/'source')]
import torch,numpy as np,pandas as pd
from hard_rule_workflow import train,evaluate,plot
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
print('Device:',torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU (full mode will be slow)')

from extension_workflow import ALL_SM_LABELS
print("Models:", ALL_SM_LABELS)
import json
from extension_workflow import Experiment,HISTORY_SPECS,SPECS
from hard_rule_workflow import references
from reuse_results import run_root,plan_reuse

## Select saved results and a new output folder
These defaults match your local folder. Keep `Run_1_results` unchanged. To resume, use the same output folder and settings: completed new-model checkpoints and prediction files are loaded automatically. An interrupted model restarts its own training; completed models are not retrained. Changing architecture, options or source code requires a fresh output folder.

In Colab, mount Drive first if using Drive paths (`from google.colab import drive; drive.mount('/content/drive')`).

In [ ]:
PREVIOUS_RUN = BUNDLE / 'Run_1_results'
OUT = BUNDLE / 'results' / 'full_hard24_add_history_v3'

PREVIOUS_RUN = run_root(PREVIOUS_RUN)
OUT = Path(OUT).resolve()
if OUT == PREVIOUS_RUN or OUT.is_relative_to(PREVIOUS_RUN):
    raise ValueError('Choose a new output folder outside Run_1_results')
previous_config = json.loads((PREVIOUS_RUN/'checkpoints/run_config.json').read_text())
previous_settings = json.loads((PREVIOUS_RUN/'evaluation/evaluation_settings.json').read_text())
N = previous_config['n']
M = previous_settings['M']
N_EVAL = previous_settings['n_eval']
SEEDS = previous_settings['seeds']
LANDMARKS = previous_settings['landmarks']
BATCH_SIZE = previous_settings['batch_size']
EPOCHS = 200
PATIENCE = 15
NEW_MODELS = list(HISTORY_SPECS)
print('Reuse:',PREVIOUS_RUN)
print('New output:',OUT)
print('Cohort:',N,'M:',M,'evaluation patients before landmark filtering:',N_EVAL)
print('Landmarks:',LANDMARKS,'sampling seeds:',SEEDS)
print('Train only:',NEW_MODELS)

## Check cache reuse before starting training
The small synthetic tests below check cache routing and rejection of mismatched data, model weights, references and settings. They do not train any model. The real-data preflight then checks your archived checkpoints, patient identities, time grids, references and predictions. Old scores are only reused with matching inputs and unchanged weights.

In [ ]:
import unittest, test_reuse_results
checks = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromModule(test_reuse_results))
if not checks.wasSuccessful(): raise RuntimeError('Saved-prediction reuse checks failed')

previous_experiment = Experiment.restore(PREVIOUS_RUN/'checkpoints',device='cpu')
if set(previous_experiment.models) != set(SPECS):
    raise ValueError('This notebook expects the completed eight-model Run 1 archive')
reuse_check = plan_reuse(previous_experiment,PREVIOUS_RUN,OUT/'evaluation',
                        previous_settings,references)
print('Validated saved predictions:',reuse_check['saved_predictions'])
print('Validated reference files:',reuse_check['reference_files'])
del previous_experiment

## New-model settings
These are the explicit defaults from the history-comparator notebook: up to 200 joint-training epochs, patience 15, and five auxiliary burn-in epochs for DynamicDeepHit. All eligible training prefixes are used after the patient split. Early stopping uses validation data; the reference curves are used only for evaluation.

These defaults are not a completed hyperparameter search. See `HISTORY_COMPARATORS.md` for the published components and the necessary SFSA adaptations.

In [ ]:
HISTORY_OPTIONS={
    'DynamicDeepHit': {
        'model_config': dict(hidden=64,layers=1,attention_hidden=64,head_hidden=64,head_layers=2,dropout=0.1),
        'loss_config': dict(rank_weight=0.1,rank_sigma=0.1,auxiliary_weight=0.1,l1_weight=1e-5,rank_reduction='per_patient'),
        'lr': 1e-3, 'batch_size': 256, 'monitor': 'concordance',
        'validation_prefix_limit': 2048, 'burn_in_epochs': 5,
    },
    'SurvTransformer': {
        'model_config': dict(width=16,heads=2,layers=3,intermediate=64,head_layers=2,dropout=0.1),
        'loss_config': dict(mortality_weight=1.,duration_weight=1.),
        'lr': 1e-3, 'batch_size': 256, 'weight_decay': 1e-4,
        'monitor': 'loss', 'validation_prefix_limit': 2048,
    },
    'SurvTransformer-Discrete': {
        'model_config': dict(width=16,heads=2,layers=3,intermediate=64,head_layers=2,dropout=0.1),
        'loss_config': dict(mortality_weight=1.,duration_weight=1.),
        'lr': 1e-3, 'batch_size': 256, 'weight_decay': 1e-4,
        'monitor': 'loss', 'validation_prefix_limit': 2048,
    },
}

## Train only the three new models
This is the time-consuming step. The eight old models and saved cohort are copied into the new folder. Their training functions are never called. The copied split ensures the new models use the same training and validation patients as the old models.

In [ ]:
experiment = train(OUT,n=N,epochs=EPOCHS,patience=PATIENCE,device=DEVICE,
                   history_options=HISTORY_OPTIONS,labels=NEW_MODELS,reuse_from=PREVIOUS_RUN)
assert set(experiment.models) == set(SPECS) | set(NEW_MODELS)
display(pd.DataFrame(experiment.config['training']).T)

## Reuse old predictions and evaluate the new models
Old direct predictions, Glazier rollouts and oracle predictions are imported, with the old model names mapped to Lite. The new models produce direct survival curves for the identical patients and time grids. M=10,000 describes the saved rollout methods; direct predictors do not use Monte Carlo trajectories.

The saved deterministic references are checked against the unchanged generator. No Monte Carlo reference strengthening or old-model sampling is repeated. Metrics include the prediction origin and a reuse flag. Old runtime measurements retain their original values and should not be treated as a same-machine runtime comparison with newly computed predictions.

In [ ]:
replicates,summary = evaluate(experiment,OUT/'evaluation',M=M,n_eval=N_EVAL,
                              seeds=SEEDS,landmarks=LANDMARKS,batch_size=BATCH_SIZE,
                              reuse_from=PREVIOUS_RUN)
assert int(replicates.reused_prediction.sum()) == reuse_check['saved_predictions']
assert not replicates.loc[replicates.model.isin(NEW_MODELS),'reused_prediction'].any()
print('Reused prediction files:',int(replicates.reused_prediction.sum()))
print('New-model prediction files:',int((~replicates.reused_prediction).sum()))
display(summary.sort_values(['landmark','MAE']))

## Combined figures and new-model learning curves
The comparison bars and mean survival-curve plots now include both old and new models. Patient-level curve errors are computed before averaging; mean curves can hide individual errors. Error bars summarize the original rollout seeds, not uncertainty across training runs.

In [ ]:
plot(OUT/'evaluation')
display(summary.loc[summary.model.isin(NEW_MODELS + ['DynamicDeepHIt-Lite','SurvTransformer-Lite','Glazier-Transformer'])]
        .sort_values(['landmark','MAE']))
for label in NEW_MODELS:
    path=OUT/'checkpoints'/(label+'_training.csv')
    if path.exists():
        print(label)
        display(pd.read_csv(path))
print('Figures:',OUT/'evaluation/plots')

## Save the combined results
The new results ZIP contains the copied cohort and old checkpoints, all three new checkpoints, training logs, old and new predictions, reference curves, combined metrics and figures, and a reuse audit. The original `Run_1_results` remains unchanged. Keep the source ZIP with this archive.

In [ ]:
import shutil
(OUT/'model_fidelity_notes.md').write_text((BUNDLE/'HISTORY_COMPARATORS.md').read_text(encoding='utf-8'),encoding='utf-8')
(OUT/'reuse_notes.txt').write_text(
    f'Previous run: {PREVIOUS_RUN}\nSaved predictions reused: {reuse_check["saved_predictions"]}\n'
    'Only DynamicDeepHit, SurvTransformer and SurvTransformer-Discrete were fitted anew.\n'
    'Old runtime measurements come from the original run and may use different hardware.\n'
    'Reused curve errors and references preserve the original observed-generator KM target.\n',encoding='utf-8')
archive = Path(shutil.make_archive(str(OUT)+'_results','zip',root_dir=OUT))
print('Combined results ZIP:',archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))